# Quickstart: classify a point cloud

This first notebook takes you from a fresh install to a class prediction on a 3D point cloud, and introduces the three ideas the rest of the library is built on:

- the **model registry** and the `create_model` factory (the `timm`-style entry point),
- the **packed-batch** tensor format (the PyTorch Geometric convention used everywhere here),
- how a model maps points to logits, and how to read the prediction back.

Every cell runs on CPU in a few seconds; the last section downloads a 6 MB checkpoint. Follow-ups: [segment a scene](02-segmentation-inference.md), [preprocessing pipelines](03-transforms.md), [your own data](04-custom-dataset.md), and [training](05-training.md).

In [ ]:
# On Colab, install the library first (uncomment):
# !pip install "torch-pointcloud[pyg-lib]"

import torch

import torch_pointcloud as tp

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| torch", torch.__version__, "| device:", device)

## Find a model in the registry

Models are built by name through a single factory, `create_model`, mirroring `timm.create_model`. Names follow the pattern `<arch>-<variant>.<dataset>`, for example `pointnet2-ssg.modelnet40.xu-yan`.

List what is available for a task with `list_models` (it accepts a glob):

In [ ]:
from torch_pointcloud.models import list_models

list_models("pointnet2*", task="classification")

## Build a model

`create_model(name, task=...)` returns a ready `nn.Module`. Two flags matter:

- `pretrained=True` loads the registered weights (cached locally). We leave it off here so the cell runs anywhere with no download: you get the architecture with random weights.
- `return_info=True` also returns the registry entry, including the exact transform pipeline the checkpoint was trained with (used in the [segmentation notebook](02-segmentation-inference.md)).

The factory returns a typed `ClassificationModel`, so `.num_classes`, `.eval()`, and `.to(device)` work as usual.

In [ ]:
model = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification").eval().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(type(model).__name__, "| classes:", model.num_classes, "| parameters:", f"{n_params:,}")

## The packed-batch format

A batch of point clouds is stored *packed*: all points concatenated into one flat tensor, with a companion `batch` vector giving each point's cloud index. Nothing is padded to a common size.

| tensor  | shape    | meaning                                                  |
| ------- | -------- | -------------------------------------------------------- |
| `pos`   | $(N, 3)$ | XYZ of every point in the batch, concatenated            |
| `x`     | $(N, C)$ | optional per-point features (`None` uses coordinates only) |
| `batch` | $(N,)$   | cloud index in $[0, B)$ for each point                   |

For $B$ clouds with $N_i$ points each, $N = N_1 + \cdots + N_B$. A reduction over one cloud becomes a `scatter` op on `batch`, never a Python loop. Let us build a batch of two toy spheres:

In [ ]:
def sphere(n: int) -> torch.Tensor:
    v = torch.randn(n, 3)
    return v / v.norm(dim=1, keepdim=True)


cloud_a, cloud_b = sphere(2048), sphere(1536)

pos = torch.cat([cloud_a, cloud_b], dim=0)
batch = torch.cat([
    torch.zeros(len(cloud_a), dtype=torch.long),
    torch.ones(len(cloud_b), dtype=torch.long),
])

print("pos:", tuple(pos.shape), "| batch:", tuple(batch.shape), "| clouds:", int(batch.max()) + 1)

Building `batch` by hand is fine for a demo; the `collate` helper does it for you (see [Use your own data](04-custom-dataset.md)). Here is a quick look at both clouds, with a small matplotlib helper we reuse across the notebooks. Passing the packed cloud its own `batch` vector as `color` is what shows the packing: one color per cloud, over points that all live in one tensor.

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax


fig = plt.figure(figsize=(9, 3.2))
show_cloud(cloud_a, ax=fig.add_subplot(131, projection="3d"), title=f"cloud A: {len(cloud_a):,} points", size=2)
show_cloud(cloud_b, ax=fig.add_subplot(132, projection="3d"), title=f"cloud B: {len(cloud_b):,} points", size=2)
show_cloud(pos, color=batch, ax=fig.add_subplot(133, projection="3d"), title=f"packed: {len(pos):,} points", size=2);

![Two toy spheres of different point counts, then the same points as one packed batch colored by batch index.](../assets/tutorials/quickstart_batch.png)

Look at the third panel: it holds exactly the points of the first two, at the coordinates they already had. Packing concatenates rows and touches nothing else, and `batch` is the only thing that says which cloud a row came from.

![Animation contrasting three ways to hold a batch of clouds: a Python list, a padded tensor, and the packed layout used here.](../assets/animations/batch_modes.webp)

A padded $(B, N, C)$ tensor would stretch cloud B up to 2,048 rows and mask the difference in every operation. The packed layout stores $N_1 + N_2$ rows and nothing else.

## Run the model

A classification model is called as `model(x, pos, batch)`: features first (here `None`), then coordinates, then the batch index. It returns one logit vector per cloud, shape $(B, \text{num\_classes})$.

In [ ]:
with torch.no_grad():
    logits = model(None, pos.to(device), batch.to(device))

print("logits:", tuple(logits.shape))  # (2, 40): one row per cloud

## Read the prediction

Turn logits into probabilities with a softmax, take the top-k, and map indices to ModelNet40 class names.

> Our weights are random, so the labels below are meaningless: they show the mechanics. Load `pretrained=True` for a real prediction.

In [ ]:
from torch_pointcloud.datasets.modelnet import MODELNET40_CLASSES

probs = logits.softmax(dim=-1)
topk = probs.topk(3, dim=-1)
for i in range(probs.shape[0]):
    names = (MODELNET40_CLASSES[j] for j in topk.indices[i].tolist())
    scores = (f"{s:.2f}" for s in topk.values[i].tolist())
    print(f"cloud {i}: " + ", ".join(f"{n} ({s})" for n, s in zip(names, scores)))

## Using real pretrained weights

The only change for a real prediction is `pretrained=True`. Adding `return_info=True` also hands back the preprocessing the checkpoint was trained with, which for this one is farthest point sampling to 1,024 points followed by `Rescale` into the unit sphere. The weights download to a local cache on first use.

Below, three of the objects committed with these docs go through that pipeline and then through the model.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
from plyfile import PlyData

from torch_pointcloud.utils.data import collate


def sample_object(name):
    """Read one object committed with these docs, downloading it when run outside a docs checkout."""
    path = Path(f"../assets/data/{name}.ply")
    if not path.exists():
        path = Path(f"{name}.ply")
        url = f"https://github.com/arthurdjn/pytorch-pointcloud/raw/main/docs/assets/data/{name}.ply"
        if not path.exists():
            urllib.request.urlretrieve(url, path)
    vertex = PlyData.read(path)["vertex"]
    return torch.from_numpy(np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32))


model, info = tp.create_model(
    "pointnet2-ssg.modelnet40.xu-yan",
    task="classification",
    pretrained=True,
    return_info=True,
)
model = model.eval()

fig = plt.figure(figsize=(9, 3.2))
probabilities = {}
for i, name in enumerate(("sample_car", "sample_chair", "sample_lamp")):
    sample = info["transform"]({"pos": sample_object(name)})  # the preprocessing this checkpoint expects
    batch = collate([sample])
    with torch.no_grad():
        probs = model(batch.get("x"), batch["pos"], batch["batch"]).softmax(dim=-1)[0]

    probabilities[name] = probs
    top = int(probs.argmax())
    show_cloud(
        sample["pos"][:, [0, 2, 1]],  # the .ply objects are y-up, plot them z-up
        ax=fig.add_subplot(1, 3, i + 1, projection="3d"),
        title=f"{MODELNET40_CLASSES[top]} ({probs[top]:.2f})",
        size=3,
    )

![Three objects, each captioned with the class the pretrained classifier gives it: car at 1.00, chair at 0.62, lamp at 1.00.](../assets/tutorials/quickstart_prediction.png)

Each panel is the 1,024 points the model actually reads, and its caption is the top of the softmax. Look at the three scores: the car and the lamp come back at 1.00, the chair does not.

The rest of the answer is what the model put on every other class, over the classes that took at least a percent:

In [ ]:
for name, probs in probabilities.items():
    ranked = [int(i) for i in probs.argsort(descending=True) if probs[i] >= 0.01]
    print(f"{name}: " + ", ".join(f"{MODELNET40_CLASSES[i]} {probs[i]:.2f}" for i in ranked))

The car and the lamp take one class each and leave nothing measurable for anything else. The chair splits: 0.62 on `chair`, then 0.22 on `stool` and 0.12 on `bench`, two classes with the same four-legged silhouette, and a percent or two on `bookshelf` and `tv_stand`. A softmax over 40 classes rarely comes back one-hot, and the runners-up tell you what the model found ambiguous.

Reproducing a checkpoint's reported accuracy means matching that preprocessing exactly, which is why `create_model(..., return_info=True)` returns it with the weights.

## Next steps

- [Segment a scene](02-segmentation-inference.md): dense per-point labels and the inferer contract.
- [Preprocessing pipelines](03-transforms.md): compose transforms step by step.
- [Use your own data](04-custom-dataset.md): build a `Dataset` and collate it into packed batches.
- [Train a model](05-training.md): an end-to-end loop with PyTorch Lightning.